# M1 데이터 확인 및 라벨 기준

M1만 사용해 normal vs efd_possible 분류 기준을 재계산한다.

In [1]:
from pathlib import Path
import csv
import io
import zipfile

import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / '05_데이터셋' / 'PreDist'
META_DIR = DATA_DIR / 'metadata' / 'manufacturer_1'
ZIP_PATH = DATA_DIR / 'predist_dataset.zip'
OUTPUT_DIR = ROOT / '07_데이터산출물'
OUTPUT_DIR.mkdir(exist_ok=True)

COMMON_SENSOR_COLUMNS = [
    'outdoor_temperature',
    's_hc1_supply_temperature',
    's_hc1_supply_temperature_setpoint',
    'p_hc1_return_temperature',
    'p_net_meter_energy',
    'p_net_meter_volume',
    'p_net_meter_heat_power',
    'p_net_meter_flow',
    'p_net_supply_temperature',
    'p_net_return_temperature',
]

def read_metadata_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(META_DIR / name, sep=';')

In [2]:
faults = read_metadata_csv('faults.csv')
normal_events = read_metadata_csv('normal_events.csv')
feature_descriptions = read_metadata_csv('feature_descriptions.csv')

strict_positive = faults[
    (faults['efd_possible'].astype(str).str.strip().str.lower() == 'true')
    & faults['Possible anomaly start'].notna()
    & faults['Possible anomaly end'].notna()
    & faults['Training start'].notna()
    & faults['Training end'].notna()
].copy()

summary = pd.DataFrame([
    {'item': 'normal_events', 'count': len(normal_events)},
    {'item': 'faults', 'count': len(faults)},
    {'item': 'strict_efd_possible', 'count': len(strict_positive)},
])
summary

,item,count
0,normal_events,35
1,faults,33
2,strict_efd_possible,15


In [3]:
with zipfile.ZipFile(ZIP_PATH) as zf:
    operational_paths = sorted(
        [name for name in zf.namelist() if name.startswith('manufacturer 1/operational_data/substation_') and name.endswith('.csv')],
        key=lambda name: int(name.rsplit('_', 1)[1].removesuffix('.csv')),
    )
    header_by_substation = {}
    for name in operational_paths:
        substation_id = int(name.rsplit('_', 1)[1].removesuffix('.csv'))
        with zf.open(name) as f:
            reader = csv.reader(io.TextIOWrapper(f, encoding='utf-8-sig'), delimiter=';')
            header = next(reader)
        header_by_substation[substation_id] = set(header) - {'timestamp'}

common_columns = sorted(set.intersection(*header_by_substation.values()))
assert common_columns == sorted(COMMON_SENSOR_COLUMNS)

pd.DataFrame({'common_sensor_column': COMMON_SENSOR_COLUMNS})

,common_sensor_column
0,outdoor_temperature
1,s_hc1_supply_temperature
2,s_hc1_supply_temperature_setpoint
3,p_hc1_return_temperature
4,p_net_meter_energy
5,p_net_meter_volume
6,p_net_meter_heat_power
7,p_net_meter_flow
8,p_net_supply_temperature
9,p_net_return_temperature


In [4]:
normal_rows = []
for _, row in normal_events.sort_values(['Event start', 'Event ID']).iterrows():
    start = pd.to_datetime(row['Event start'])
    end = pd.to_datetime(row['Event end'])
    normal_rows.append({
        'sample_id': f"normal_{int(row['Event ID']):04d}",
        'manufacturer': 'manufacturer_1',
        'label': 'normal',
        'source_file': 'normal_events.csv',
        'source_event_id': int(row['Event ID']),
        'substation_id': int(row['substation ID']),
        'window_start': start.strftime('%Y-%m-%d %H:%M:%S'),
        'window_end': end.strftime('%Y-%m-%d %H:%M:%S'),
        'window_days': round((end - start).total_seconds() / 86400, 6),
        'window_policy': 'normal_event_start_to_end',
        'report_date': '',
        'possible_anomaly_start': '',
        'possible_anomaly_end': '',
        'training_start': row['Training start'],
        'training_end': row['Training end'],
    })

positive_rows = []
for _, row in strict_positive.sort_values(['Report date', 'Event ID']).iterrows():
    report_date = pd.to_datetime(row['Report date'])
    start = report_date - pd.Timedelta(days=7)
    positive_rows.append({
        'sample_id': f"efd_possible_{int(row['Event ID']):04d}",
        'manufacturer': 'manufacturer_1',
        'label': 'efd_possible',
        'source_file': 'faults.csv',
        'source_event_id': int(row['Event ID']),
        'substation_id': int(row['substation ID']),
        'window_start': start.strftime('%Y-%m-%d %H:%M:%S'),
        'window_end': report_date.strftime('%Y-%m-%d %H:%M:%S'),
        'window_days': 7.0,
        'window_policy': 'report_date_minus_7d_to_report_date',
        'report_date': report_date.strftime('%Y-%m-%d %H:%M:%S'),
        'possible_anomaly_start': row['Possible anomaly start'],
        'possible_anomaly_end': row['Possible anomaly end'],
        'training_start': row['Training start'],
        'training_end': row['Training end'],
    })

event_index = pd.DataFrame(normal_rows + positive_rows).sort_values(['label', 'window_start', 'source_event_id']).reset_index(drop=True)
event_index.to_csv(OUTPUT_DIR / 'm1_classification_event_index.csv', index=False, encoding='utf-8-sig')
event_index.groupby('label').size().rename('count').reset_index()

,label,count
0,efd_possible,15
1,normal,35


In [5]:
assert len(event_index) == 50
assert event_index['label'].value_counts().to_dict() == {'normal': 35, 'efd_possible': 15}
assert set(event_index['manufacturer']) == {'manufacturer_1'}
assert set(event_index['window_days'].round(6)) == {7.0}
event_index.head()

,sample_id,manufacturer,label,source_file,source_event_id,substation_id,window_start,window_end,window_days,window_policy,report_date,possible_anomaly_start,possible_anomaly_end,training_start,training_end
0,efd_possible_0001,manufacturer_1,efd_possible,faults.csv,1,10,2014-04-27 14:44:00,2014-05-04 14:44:00,7.0,report_date_minus_7d_to_report_date,2014-05-04 14:44:00,2014-05-03 16:00:00,2014-05-05 04:00:00,2012-03-28 09:00:00,2014-04-20 14:44:00
1,efd_possible_0003,manufacturer_1,efd_possible,faults.csv,3,12,2015-11-24 10:56:00,2015-12-01 10:56:00,7.0,report_date_minus_7d_to_report_date,2015-12-01 10:56:00,2015-11-29 12:00:00,2015-12-02 10:56:00,2015-03-01 00:00:00,2015-11-17 10:56:00
2,efd_possible_0040,manufacturer_1,efd_possible,faults.csv,40,24,2016-03-31 07:47:00,2016-04-07 07:47:00,7.0,report_date_minus_7d_to_report_date,2016-04-07 07:47:00,2016-04-05 13:00:00,2016-04-08 07:47:00,2015-11-20 00:00:00,2016-03-24 07:47:00
3,efd_possible_0052,manufacturer_1,efd_possible,faults.csv,52,21,2016-12-05 15:55:00,2016-12-12 15:55:00,7.0,report_date_minus_7d_to_report_date,2016-12-12 15:55:00,2016-12-11 20:00:00,2016-12-13 15:55:00,2015-11-30 09:00:00,2016-11-28 15:55:00
4,efd_possible_0067,manufacturer_1,efd_possible,faults.csv,67,7,2017-11-19 13:15:00,2017-11-26 13:15:00,7.0,report_date_minus_7d_to_report_date,2017-11-26 13:15:00,2017-04-23 01:00:00,2017-11-27 13:15:00,2015-04-12 00:00:00,2017-04-11 00:00:00
